In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
)


project_root = Path.cwd().parent

data_path = (
    project_root
    / "data"
    / "processed"
    / "vic_demand_2024-07_to_2026-06.parquet"
)

df = pd.read_parquet(data_path)

df = df[
    df["SETTLEMENTDATE"] < pd.Timestamp("2026-07-01")
].copy()

df = df.sort_values("SETTLEMENTDATE").reset_index(drop=True)

df.head()

,REGION,SETTLEMENTDATE,TOTALDEMAND,RRP,PERIODTYPE
0,VIC1,2024-07-01 00:05:00,5404.13,281.32,TRADE
1,VIC1,2024-07-01 00:10:00,5346.60,275.61,TRADE
2,VIC1,2024-07-01 00:15:00,5240.84,270.98,TRADE
3,VIC1,2024-07-01 00:20:00,5239.06,270.98,TRADE
4,VIC1,2024-07-01 00:25:00,5226.10,270.98,TRADE


In [2]:
FORECAST_HORIZON = 6

df["target_30m"] = df["TOTALDEMAND"].shift(-FORECAST_HORIZON)

df[
    [
        "SETTLEMENTDATE",
        "TOTALDEMAND",
        "target_30m",
    ]
].head(10)

,SETTLEMENTDATE,TOTALDEMAND,target_30m
0,2024-07-01 00:05:00,5404.13,5198.55
1,2024-07-01 00:10:00,5346.60,5163.32
2,2024-07-01 00:15:00,5240.84,5150.10
3,2024-07-01 00:20:00,5239.06,5158.83
4,2024-07-01 00:25:00,5226.10,5135.86
5,2024-07-01 00:30:00,5237.71,5012.65
6,2024-07-01 00:35:00,5198.55,5013.21
7,2024-07-01 00:40:00,5163.32,5025.48
8,2024-07-01 00:45:00,5150.10,5007.50
9,2024-07-01 00:50:00,5158.83,4990.78


In [3]:
df["hour"] = df["SETTLEMENTDATE"].dt.hour
df["minute"] = df["SETTLEMENTDATE"].dt.minute
df["day_of_week"] = df["SETTLEMENTDATE"].dt.dayofweek
df["month"] = df["SETTLEMENTDATE"].dt.month
df["is_weekend"] = df["day_of_week"] >= 5

In [4]:
df["baseline_current"] = df["TOTALDEMAND"]

df["baseline_yesterday"] = (
    df["TOTALDEMAND"]
    .shift(288 - FORECAST_HORIZON)
)

df["baseline_last_week"] = (
    df["TOTALDEMAND"]
    .shift(2016 - FORECAST_HORIZON)
)

In [5]:
model_df = df.dropna(
    subset=[
        "target_30m",
        "baseline_current",
        "baseline_yesterday",
        "baseline_last_week",
    ]
).copy()

train = model_df[
    model_df["SETTLEMENTDATE"] < "2026-03-01"
].copy()

validation = model_df[
    (model_df["SETTLEMENTDATE"] >= "2026-03-01")
    & (model_df["SETTLEMENTDATE"] < "2026-05-01")
].copy()

test = model_df[
    (model_df["SETTLEMENTDATE"] >= "2026-05-01")
    & (model_df["SETTLEMENTDATE"] < "2026-07-01")
].copy()

print(f"Train:      {len(train):,}")
print(f"Validation: {len(validation):,}")
print(f"Test:       {len(test):,}")

Train:      173,093
Validation: 17,568
Test:       17,562


In [6]:
def evaluate_forecast(
    data: pd.DataFrame,
    prediction_column: str,
) -> dict:
    actual = data["target_30m"]
    predicted = data[prediction_column]

    mae = mean_absolute_error(actual, predicted)

    rmse = np.sqrt(
        mean_squared_error(actual, predicted)
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
    }


baseline_results = pd.DataFrame(
    {
        "Current demand": evaluate_forecast(
            validation,
            "baseline_current",
        ),
        "Same time yesterday": evaluate_forecast(
            validation,
            "baseline_yesterday",
        ),
        "Same time last week": evaluate_forecast(
            validation,
            "baseline_last_week",
        ),
    }
).T

baseline_results

,MAE,RMSE
Current demand,163.659187,203.675770
Same time yesterday,400.329073,591.630755
Same time last week,480.694785,696.469901


### 30-minute baseline performance

For a 30-minute forecasting horizon, the persistence forecast — assuming
demand in 30 minutes will equal current demand — is substantially stronger
than calendar-based historical baselines.

Validation performance:

- Current demand: MAE = 163.7 MW, RMSE = 203.7 MW
- Same time yesterday: MAE = 400.3 MW, RMSE = 591.6 MW
- Same time last week: MAE = 480.7 MW, RMSE = 696.5 MW

The strength of the persistence benchmark is consistent with the very high
short-term autocorrelation observed in electricity demand.

Any learned forecasting model should therefore be evaluated against the
163.7 MW persistence MAE rather than merely against historical-day
benchmarks.